# Imports

In [1]:
import sys, os
os.environ['MKL_THREADING_LAYER'] = 'GNU'

sys.path.insert(0, os.path.abspath('..'))
from src.utils import *

# Loading Data

In [2]:
# load the data for both mfa and normal annotations 
# dictionary structure is patient_data[patient][position][method][hg_trace, hg_map, phon_seq]
# you can access the data as follows: patient_data['S14']['p1']['Kumar']['hg_trace'] etc. 
path_to_intraop_data = r"C:\Users\Nabiya\Box\Academic-Duke\CoganLab\Summer 2024\intraop\ieeg_data_intraop\Zac_intraop_data"
patient_data = fetch_patient_data("../data/processed/patient_data.pkl", path_to_intraop_data)


Loaded patient_data from pickle.


# Useful Dictionaries and Functions

In [3]:
numtoPhon = {1:'a', 2:'ae', 3:'i', 4:'u', 5:'b', 6:'p', 7:'v', 8:'g', 9:'k'}
phoneme_group = {
        'a': 'low', 'ae': 'low', 'i': 'high', 'u': 'high',
        'b': 'labial', 'p': 'labial', 'v': 'labial',
        'g': 'dorsal', 'k': 'dorsal'
    }

patients = ['S14', 'S26']
positions =['p1', 'p2', 'p3', 'all']
methods = ['Kumar', 'MFA']

In [4]:
patient_data['S14']['p1']["Kumar"]['hg_trace'].shape

(144, 200, 111)

# tSNE Latent Space Representation (Panels C and D)

In [5]:
# plot 50 tsne latent space repr. (only if silscore > 0.25 do we save fig)
Xys_df_km = get_training_data(patient_data, 'S26', 'p1', 'Kumar')
Xys_df_mfa = get_training_data(patient_data, 'S26', 'p1', 'MFA')
if False:
    for i in range(50): 
            for i in range(1,21):
                tsne_df_km, pca_ref = run_tsne(Xys_df_km, 50, 0.8, i)
                tsne_df_mfa, _ = run_tsne(Xys_df_mfa, 50, 0.8, i, pca_ref=pca_ref)
                
                run_plottly(tsne_df_km, 'Phoneme')
                run_plottly(tsne_df_mfa, 'Phoneme')

# Silhoutte Score Distributions (Panels E and F)

In [6]:
pkl_path_silscores = r"../data/processed/silscores_allPatients.pkl"
silscores = fetch_silscore_data(patient_data, pkl_path_silscores)

c:\Users\Nabiya\anaconda3\envs\ADA_HW\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


KeyboardInterrupt: 

## Restructure Dataframe for plotting/analyses

In [8]:
silscores_df = silscores_to_df(silscores)
silscores_df.to_csv("../data/results/silscores_flattened.csv", index=False)

In [9]:
# rows = []
# for patient, methods_dict in silscores.items():
#     for method, scores_dict in methods_dict.items():
#         for score_type, values in scores_dict.items():
#             for i, val in enumerate(values, start=1):
#                 rows.append({
#                     "Patient": patient,
#                     "Method": method,
#                     "Score_Type": score_type,
#                     "Iteration": i,
#                     "Value": val
#                 })
# silscores_df = pd.DataFrame(rows)
# # Optional: clean up column order or sort
# silscores_df = silscores_df[["Patient", "Method", "Score_Type", "Iteration", "Value"]]
# silscores_df.sort_values(by=["Patient", "Method", "Score_Type", "Iteration"], inplace=True)
# # Now you can save or inspect


In [10]:
silscores_df = pd.read_csv("../data/results/silscores_flattened.csv")
silscores_df['Method'] = silscores_df['Method'].replace(
    {'Kumar': "Manual", 'MFA': 'Automated'}
)
silscores_df['Score_Type'] = silscores_df['Score_Type'].replace(
    {'Silhoutte_Score_PhonType': "Vowel/Consonant",
    'Silhoutte_Score_PhonGrp': 'Articulatory Group', 
    'Silhoutte_Score_Phon': 'True Distribution', 
    'Silhouette_Score_PhonChance': 'Chance Distribution'
    }
)

silscores_df['Patient'] = silscores_df['Patient'].replace(
    {'S26': 'S1',
    'S23': 'S2',
    'S33': 'S3',
    'S22': 'S4'}
)
silscores_df = silscores_df.rename(columns={
    "Score_Type": "Silhouette Level",
    "Value": "Silhouette Score"
})
    
silscores_df.head()

,Patient,Method,Silhouette Level,Iteration,Silhouette Score
0,S4,Manual,Chance Distribution,1,0.160615
1,S4,Manual,Chance Distribution,2,0.060780
2,S4,Manual,Chance Distribution,3,0.045845
3,S4,Manual,Chance Distribution,4,0.036651
4,S4,Manual,Chance Distribution,5,0.063764


## Plots

In [ ]:
colorsmap = {
    'Manual': px.colors.qualitative.Vivid[1],
    'Automated': px.colors.qualitative.Vivid[0]
}
# plot order true first then chance
orderforplot = {
    "Difference_Variable": ['Onset', 'Offset', 'Duration'], 
    "Phoneme": ['a', 'ae', 'b', 'g', 'i', 'k', 'p', 'u', 'v'], 
    "Patient": ['S1', 'S2', 'S3', 'S4'],
    'Silhouette Level':['True Distribution', 'Chance Distribution'] }

silscor_dfmelted_tr_ch = silscores_df[(silscores_df['Silhouette Level'] == 'True Distribution') | (silscores_df['Silhouette Level'] == 'Chance Distribution')]


fig = px.box(
    silscor_dfmelted_tr_ch,
    y='Silhouette Score',
    x='Silhouette Level',
    color='Method',
    points='all',  # Shows stripplot-like points
    boxmode='group',
    color_discrete_sequence= px.colors.qualitative.Vivid, 
    color_discrete_map= colorsmap, 
    category_orders=orderforplot,
    hover_name='Silhouette Level', 
    hover_data= ['Patient', 'Method', 'Silhouette Level', 'Iteration',
       'Silhouette Score']    
    )
fig.update_traces(marker=dict(size=4,opacity=0.9), jitter=0.3, boxmean=True)

# Add significance annotations manually
levels = silscor_dfmelted_tr_ch['Silhouette Level'].unique()

for i, level in enumerate(levels):
    sub_df = silscor_dfmelted_tr_ch[silscor_dfmelted_tr_ch['Silhouette Level'] == level]
    scores_manual = sub_df[sub_df['Method'] == 'Manual']['Silhouette Score'].to_numpy()
    scores_auto = sub_df[sub_df['Method'] == 'Automated']['Silhouette Score'].to_numpy()
    
    stat, p_val = wilcoxon(scores_manual, scores_auto)

    # Define label
    sig_label = 'ns'
    if p_val < 0.001:
        sig_label = '***'
    elif p_val < 0.01:
        sig_label = '**'
    elif p_val < 0.05:
        sig_label = '*'

    # Compute annotation Y position
    y_max = sub_df['Silhouette Score'].max()
    fig.add_trace(go.Scatter(
        x=[level],
        y=[y_max + 0.01],
        mode='text',
        text=[sig_label],
        textposition='top center',
        showlegend=False
    ))


mm_to_px = 3.78
fig.update_layout(
    title=dict(text="TSNE Latent Space Silhouette Scores (All patients)", font=dict(size=12)),
    boxmode="group",
    yaxis_title=dict(text="Silhouette Score", font=dict(size=10)),
    xaxis_title="",
    legend_title=dict(text="Method", font=dict(size=10)),
    template='simple_white',
    width=200 * mm_to_px,   
    height=150 * mm_to_px,  
    font=dict(size=10)
)
# plt.savefig("../figures/figure3/manualvsautomated_TSNEsilhouetteS26.svg", format='svg')
#fig.write_image("../figures/figure3/manualvsautomated_TSNEsilhouetteS26PhonChancePlotly.pdf", format='pdf')
fig.show()

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
sns.reset_defaults()


# extract True and Chance subsets
true_df = silscor_dfmelted_tr_ch[silscor_dfmelted_tr_ch['Silhouette Level'] == 'True Distribution']
chance_df = silscor_dfmelted_tr_ch[silscor_dfmelted_tr_ch['Silhouette Level'] == 'Chance Distribution']

# compute mean chance silhouette score per Patient × Method
chance_means = chance_df.groupby(['Patient', 'Method'], as_index=False)['Silhouette Score'].mean()

# main plot — true scores only
fig = px.box(
    true_df,
    x='Patient',
    y='Silhouette Score',
    color='Method',
    points='all',
    boxmode='group',
    color_discrete_map=colorsmap,
    hover_name='Patient',
    hover_data=['Patient', 'Method', 'Iteration', 'Silhouette Score'],
    category_orders=orderforplot
    
)

fig.update_traces(marker=dict(size=4, opacity=0.9), jitter=0.3, boxmean=False)

# map each (Patient, Method) combo to x coordinate
x_coords = {p: i for i, p in enumerate(true_df['Patient'].unique())}
method_offset = {'Manual': -0.2, 'Automated': 0.2}  # horizontal shift for group boxes

# add horizontal chance lines beneath each box
for _, row in chance_means.iterrows():
    patient = row['Patient']
    method = row['Method']
    chance_val = row['Silhouette Score']
    x = x_coords[patient] + method_offset[method]

    fig.add_shape(
        type="line",
        x0=x - 0.15,
        x1=x + 0.15,
        y0=chance_val,
        y1=chance_val,
        line=dict(color="gray", dash="dash"),
        xref="x",
        yref="y"
    )

mm_to_px = 3.78
fig.update_layout(
    title=dict(text="t-SNE Silhouette Scores per Patient", font=dict(size=12)),
    boxmode="group",
    yaxis_title=dict(text="Silhouette Score", font=dict(size=10)),
    xaxis_title="Patient",
    legend_title=dict(text="Method", font=dict(size=10)),
    font=dict(size=10))
fig.write_image("../figures/figure3/manualvsautomated_TSNEsilhouetteAllpatientsphonChancePlotly.svg", format='svg', scale=3)
fig.show()


In [ ]:
silscor_dfmelted2S1 = silscor_dfmelted_tr_ch[silscor_dfmelted_tr_ch['Patient'] == 'S1']
fig = px.box(
    silscor_dfmelted2S1,
    y='Silhouette Score',
    x='Silhouette Level',
    color='Method',
    points='all',  # Shows stripplot-like points
    boxmode='group',
    color_discrete_sequence= px.colors.qualitative.Vivid, 
    color_discrete_map= colorsmap, 
    category_orders=orderforplot    
    )
#fig.update_traces(marker=dict(size=4,opacity=0.9), jitter=0.3, boxmean=True)

# Add significance annotations manually
levels = silscor_dfmelted2S1['Silhouette Level'].unique()

for i, level in enumerate(levels):
    sub_df = silscor_dfmelted2S1[silscor_dfmelted2S1['Silhouette Level'] == level]
    scores_manual = sub_df[sub_df['Method'] == 'Manual']['Silhouette Score'].to_numpy()
    scores_auto = sub_df[sub_df['Method'] == 'Automated']['Silhouette Score'].to_numpy()
    
    stat, p_val = mannwhitneyu(scores_manual, scores_auto)

    # Define label
    sig_label = 'ns'
    if p_val < 0.001:
        sig_label = '***'
    elif p_val < 0.01:
        sig_label = '**'
    elif p_val < 0.05:
        sig_label = '*'

    # Compute annotation Y position
    y_max = sub_df['Silhouette Score'].max()
    fig.add_trace(go.Scatter(
        x=[level],
        y=[y_max + 0.01],
        mode='text',
        text=[sig_label],
        textposition='top center',
        showlegend=False
    ))


mm_to_px = 3.78
fig.update_layout(
    title=dict(text="TSNE Latent Space Silhouette Scores", font=dict(size=12)),
    boxmode="group",
    yaxis_title=dict(text="Silhouette Score", font=dict(size=10)),
    xaxis_title="",
    legend_title=dict(text="Method", font=dict(size=10)),
    font=dict(size=10)
)
# plt.savefig("../figures/figure3/manualvsautomated_TSNEsilhouetteS26.svg", format='svg')
fig.write_image("../figures/figure3/manualvsautomated_TSNEsilhouetteS26PhonChancePlotly.svg", format='svg', scale=3)
fig.show()

In [ ]:
from scipy.stats import wilcoxon, mannwhitneyu

true_df = silscor_dfmelted_tr_ch[silscor_dfmelted_tr_ch['Silhouette Level'] == 'True Distribution']
chance_df = silscor_dfmelted_tr_ch[silscor_dfmelted_tr_ch['Silhouette Level'] == 'Chance Distribution']

patients = sorted(true_df['Patient'].unique())

print("---- Manual vs Automated (True only) ----")
for patient in patients:
    sub_df = true_df[true_df['Patient'] == patient]
    m = sub_df[sub_df['Method'] == 'Manual']['Silhouette Score'].to_numpy()
    a = sub_df[sub_df['Method'] == 'Automated']['Silhouette Score'].to_numpy()
    if len(m) and len(a):
        stat, p = mannwhitneyu(m, a)
        print(f"{patient}: W={stat:.3f}, p={p:.4f}")
        
print("---- Manual vs Automated (Chance only) ----")
for patient in patients:
    sub_df = chance_df[chance_df['Patient'] == patient]
    m = sub_df[sub_df['Method'] == 'Manual']['Silhouette Score'].to_numpy()
    a = sub_df[sub_df['Method'] == 'Automated']['Silhouette Score'].to_numpy()
    if len(m) and len(a):
        stat, p = mannwhitneyu(m, a)
        print(f"{patient}: W={stat:.3f}, p={p:.4f}")

print("\n---- True vs Chance (per method) ----")
methods = ['Manual', 'Automated']
for method in methods:
    for patient in patients:
        t = true_df[(true_df['Patient'] == patient) & (true_df['Method'] == method)]['Silhouette Score'].to_numpy()
        c = chance_df[(chance_df['Patient'] == patient) & (chance_df['Method'] == method)]['Silhouette Score'].to_numpy()
        t = t[~np.isnan(t)]
        c = c[~np.isnan(c)]
        if len(t) and len(c):
            stat, p = mannwhitneyu(t, c, alternative='greater')
            print(f"{patient} | {method}: U={stat:.3f}, p={p:.4f}")
